# E3D — Semantic–Collaborative Agreement Analysis (RQ2)
**SNA Group 14 · Amazon · Google Colab · read-only**

Trả lời vế 2 RQ2: *Hiệu quả semantic alignment có thay đổi theo mức độ đồng thuận giữa semantic information và collaborative information không?*

**Không train lại, không yêu cầu CSV thủ công.** Đọc trực tiếp `usr_emb_np.pkl`, `itm_emb_np.pkl`, `trn_mat.pkl`, `tst_mat.pkl` và Top-K đã xuất trong `Data_SNA_Group14`.

**Định nghĩa agreement (proxy độc lập với mô hình):** Với mỗi user/item, so sánh thứ tự độ tương đồng với một tập anchor cùng loại theo (i) semantic embedding gốc và (ii) lịch sử tương tác train-only (cosine trên ma trận tương tác). Agreement = Spearman correlation giữa hai dãy similarity. *Không* tính cosine trực tiếp giữa semantic và graph embedding khác không gian. User agreement đo trực tiếp; item agreement được tổng hợp theo các item mà user đã tương tác trong train. Các trường hợp collaborative similarity không có phương sai được gắn `NaN` và loại khỏi phân nhóm.

**Lưu ý:** Đây là operationalization của *agreement*, không phải phép đo chính thức được định nghĩa bởi paper. Nó phản ánh tính tương hợp của cấu trúc láng giềng, không chứng minh quan hệ nhân quả. Semantic profiles nguồn gốc chưa được xác nhận hoàn toàn train-only; không tuyên bố loại trừ hoàn toàn leakage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp, pickle, json, math
from scipy.stats import rankdata
from IPython.display import display
ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA = ROOT/'data/raw/amazon'
OUT = ROOT/'experiments/E3D/agreement_analysis'
SEEDS = (2023, 2024, 2025)
N_USERS, N_ITEMS, N_TEST_USERS = 11000, 9332, 9860
N_ANCHORS = 256
ANCHOR_SEED = 2026
assert ROOT.is_dir(), f'Không thấy {ROOT}'
for name in ['trn_mat.pkl','tst_mat.pkl','usr_emb_np.pkl','itm_emb_np.pkl']:
    assert (DATA/name).is_file(), f'Thiếu {DATA/name}'
print('ROOT:',ROOT)
print('Output:',OUT)


## 1. Load dữ liệu và kiểm tra kích thước

In [ ]:
def read_pickle(path):
    # Chỉ đọc pickle thuộc Drive của nhóm mà bạn tin cậy.
    with path.open('rb') as f: return pickle.load(f)
def as_matrix(x, shape):
    if sp.issparse(x): a=x.tocsr()
    else: a=sp.csr_matrix(np.asarray(x))
    assert a.shape==shape, (a.shape,shape)
    a.sum_duplicates();a.eliminate_zeros();a.data=np.ones(a.nnz,dtype=np.float32)
    a.sort_indices()
    return a
def as_emb(x,n,name):
    if hasattr(x,'detach'): x=x.detach().cpu().numpy()
    if isinstance(x,dict):
        raise TypeError(f'{name}: embedding là dict; cần kiểm tra schema gốc, không tự suy đoán thứ tự ID')
    x=np.asarray(x)
    assert x.ndim==2 and x.shape[0]==n, (name,x.shape,n)
    assert np.issubdtype(x.dtype,np.number) and np.isfinite(x).all(), name
    x=x.astype(np.float32)
    norm=np.linalg.norm(x,axis=1)
    assert np.all(norm>0), f'{name} chứa zero-norm rows'
    return x/norm[:,None]
trn=as_matrix(read_pickle(DATA/'trn_mat.pkl'),(N_USERS,N_ITEMS))
tst=as_matrix(read_pickle(DATA/'tst_mat.pkl'),(N_USERS,N_ITEMS))
user_sem=as_emb(read_pickle(DATA/'usr_emb_np.pkl'),N_USERS,'user semantic')
item_sem=as_emb(read_pickle(DATA/'itm_emb_np.pkl'),N_ITEMS,'item semantic')
test_users=np.flatnonzero(np.diff(tst.indptr)>0)
assert len(test_users)==N_TEST_USERS
assert trn.multiply(tst).nnz==0, 'Train/Test overlap!'
print('Shapes:',trn.shape,user_sem.shape,item_sem.shape,'test users:',len(test_users))


## 2. Tính semantic–collaborative agreement

Mỗi thực thể được so với **256 anchors cố định** (random seed 2026). Tính cosine similarity trong từng không gian *cùng loại* và Spearman correlation giữa hai vector similarities. Đối với collaborative similarity, dùng train interactions, không dùng validation/test. Loại self-anchor khi thực thể nằm trong tập anchors. Nếu tất cả similarity của một thực thể bằng nhau, agreement không xác định.


In [ ]:
def agreement_to_anchors(semantic, collab_csr, n_anchors=256, seed=2026, batch=512):
    n=semantic.shape[0]
    assert collab_csr.shape[0]==n
    rng=np.random.default_rng(seed)
    norms=np.sqrt(np.asarray(collab_csr.multiply(collab_csr).sum(axis=1)).ravel())
    eligible=np.flatnonzero(norms>0)
    assert len(eligible)>n_anchors, "Không đủ anchors có train interactions"
    anchors=np.sort(rng.choice(eligible,size=n_anchors,replace=False))
    # Semantic dot product = cosine vì embeddings đã chuẩn hóa.
    sem_anchors=semantic[anchors].T
    collab_anchors=collab_csr[anchors].T
    scores=np.full(n,np.nan,dtype=np.float64)
    for start in range(0,n,batch):
        stop=min(start+batch,n)
        s=semantic[start:stop]@sem_anchors
        c=(collab_csr[start:stop]@collab_anchors).toarray()
        c=c/np.maximum(norms[start:stop,None]*norms[anchors][None,:],1e-12)
        # Remove self-similarity from BOTH spaces (if entity is an anchor).
        ix=np.searchsorted(anchors,np.arange(start,stop))
        for local,(i,j) in enumerate(zip(range(start,stop),ix)):
            if j<len(anchors) and anchors[j]==i:
                s[local,j]=np.nan;c[local,j]=np.nan
        for local in range(stop-start):
            ok=np.isfinite(s[local]) & np.isfinite(c[local])
            if ok.sum()<8:continue
            a,b=s[local,ok],c[local,ok]
            if np.ptp(a)<1e-10 or np.ptp(b)<1e-10:continue
            ra,rb=rankdata(a),rankdata(b)
            ra-=ra.mean();rb-=rb.mean()
            denom=np.linalg.norm(ra)*np.linalg.norm(rb)
            if denom>1e-12:scores[start+local]=float(ra@rb/denom)
        print(f'{stop}/{n} processed',end='\r')
    print()
    return scores,anchors
# User collaborative similarity: shared train items.
user_agreement,user_anchors=agreement_to_anchors(user_sem,trn,N_ANCHORS,ANCHOR_SEED)
# Item collaborative similarity: shared train users.
item_agreement,item_anchors=agreement_to_anchors(item_sem,trn.T.tocsr(),N_ANCHORS,ANCHOR_SEED+1)
# Aggregate item-side agreement for each user using ONLY their training items.
item_agreement_filled=np.nan_to_num(item_agreement,nan=0.0)
valid_item=np.isfinite(item_agreement).astype(np.float64)
item_sum=trn@item_agreement_filled
item_count=trn@valid_item
user_item_agreement=np.divide(item_sum,item_count,out=np.full(N_USERS,np.nan),where=item_count>0)
print('Valid user-side:',np.isfinite(user_agreement).sum(), '/',N_USERS)
print('Valid item-side:',np.isfinite(item_agreement).sum(), '/',N_ITEMS)
print('Valid aggregated item-side:',np.isfinite(user_item_agreement).sum(), '/',N_USERS)


## 3. Chia nhóm Low / Medium / High theo phân vị, độc lập với kết quả test

In [ ]:
def group_by_tertile(values, ids):
    vals=values[ids]
    valid=np.isfinite(vals)
    assert valid.sum()>=300, f'Quá ít thực thể có agreement xác định: {valid.sum()}'
    q=np.quantile(vals[valid],[1/3,2/3])
    assert q[0]<q[1], f'Agreement quantiles bị trùng: {q}'
    labels=np.full(len(ids),'Undefined',dtype=object)
    labels[valid & (vals<=q[0])]='Low'
    labels[valid & (vals>q[0]) & (vals<=q[1])]='Medium'
    labels[valid & (vals>q[1])]='High'
    return labels,q
user_group,uq=group_by_tertile(user_agreement,test_users)
item_group,iq=group_by_tertile(user_item_agreement,test_users)
groups=pd.DataFrame({'user_id':test_users,'user_agreement':user_agreement[test_users],
                     'item_agreement_mean_train':user_item_agreement[test_users],
                     'user_group':user_group,'item_group':item_group,
                     'train_degree':np.diff(trn.indptr)[test_users]})
display(groups.groupby(['user_group','item_group']).size().unstack(fill_value=0))
print('User quantiles:',uq,'Item quantiles:',iq)


## 4. Đọc trực tiếp Top-K của LightGCN, Full, User-only, Item-only và đánh giá theo user

In [ ]:
RUNS=[]
for seed in SEEDS:
    RUNS.extend([
        ('LightGCN',seed,ROOT/f'experiments/predictions/lightgcn/seed_{seed}/topk_test.npz'),
        ('RLMRec-Con Full',seed,ROOT/f'experiments/predictions/lightgcn_plus/seed_{seed}/topk_test.npz'),
        ('User-only',seed,ROOT/f'experiments/E3/runs/user_only/weight_e1_default/seed_{seed}/topk/topk_test.npz'),
        ('Item-only',seed,ROOT/f'experiments/E3/runs/item_only/weight_e1_default/seed_{seed}/topk/topk_test.npz'),
    ])
missing=[str(p) for _,_,p in RUNS if not p.is_file()]
if missing: raise FileNotFoundError('Thiếu Top-K đã export trên Drive:\n'+'\n'.join(missing))
def per_user_metrics(path,k=20):
    with np.load(path,allow_pickle=False) as z:
        users=z['user_ids'].astype(np.int64)
        items=z['item_ids'].astype(np.int64)
    assert users.ndim==1 and len(users)==N_TEST_USERS and np.array_equal(np.sort(users),test_users),str(path)
    assert items.shape[0]==len(users) and items.shape[1]>=k,str(path)
    rec=items[:,:k]
    assert ((rec>=0)&(rec<N_ITEMS)).all()
    assert np.all(np.diff(np.sort(rec,axis=1),axis=1)!=0),'Duplicate items'
    assert not np.asarray(trn[np.repeat(users,k),rec.ravel()]).ravel().any(),'Train-mask collision'
    hit=np.asarray(tst[np.repeat(users,k),rec.ravel()]).reshape(-1,k).astype(bool)
    npos=np.diff(tst.indptr)[users]
    assert np.all(npos>0)
    recall=hit.sum(axis=1)/npos
    discount=1/np.log2(np.arange(2,k+2))
    dcg=hit@discount
    prefix=np.r_[0,np.cumsum(discount)]
    ndcg=dcg/prefix[np.minimum(npos,k)]
    return pd.DataFrame({'user_id':users,'Recall@20':recall,'NDCG@20':ndcg})
rows=[]
for model,seed,path in RUNS:
    x=per_user_metrics(path)
    x['model']=model;x['seed']=seed
    rows.append(x)
    print(model,seed,'Recall@20=',round(x['Recall@20'].mean(),6),'NDCG@20=',round(x['NDCG@20'].mean(),6))
per_user=pd.concat(rows,ignore_index=True).merge(groups,on='user_id',validate='many_to_one')
assert len(per_user)==12*N_TEST_USERS


## 5. So sánh delta cùng user, cùng seed

Bảng delta được tính **paired** giữa mỗi biến thể và LightGCN trên cùng user/seed. Ngoài trung bình nhóm, xuất tương quan giữa agreement liên tục và gain. Kết quả là quan hệ quan sát, không khẳng định quan hệ nhân quả.


In [ ]:
base=per_user[per_user.model=='LightGCN'][['user_id','seed','Recall@20','NDCG@20']].rename(columns={'Recall@20':'baseline_recall','NDCG@20':'baseline_ndcg'})
comparison=per_user[per_user.model!='LightGCN'].merge(base,on=['user_id','seed'],validate='many_to_one')
comparison['DeltaRecall@20']=comparison['Recall@20']-comparison['baseline_recall']
comparison['DeltaNDCG@20']=comparison['NDCG@20']-comparison['baseline_ndcg']
summary_rows=[]
for axis,col in [('User-side','user_group'),('Item-side','item_group')]:
    for (model,seed,group),g in comparison.groupby(['model','seed',col]):
        if group=='Undefined':continue
        summary_rows.append({'axis':axis,'model':model,'seed':seed,'agreement_group':group,
            'n_users':len(g),'Recall@20':g['Recall@20'].mean(),'NDCG@20':g['NDCG@20'].mean(),
            'BaselineRecall@20':g['baseline_recall'].mean(),'BaselineNDCG@20':g['baseline_ndcg'].mean(),
            'DeltaRecall@20':g['DeltaRecall@20'].mean(),'DeltaNDCG@20':g['DeltaNDCG@20'].mean()})
seed_summary=pd.DataFrame(summary_rows)
group_summary=seed_summary.groupby(['axis','model','agreement_group'],as_index=False).agg(
    n_users=('n_users','first'),
    DeltaRecall_mean=('DeltaRecall@20','mean'),
    DeltaRecall_sd=('DeltaRecall@20','std'),
    DeltaNDCG_mean=('DeltaNDCG@20','mean'),
    DeltaNDCG_sd=('DeltaNDCG@20','std'),
    Recall_mean=('Recall@20','mean'),
    NDCG_mean=('NDCG@20','mean'))
display(group_summary.sort_values(['axis','model','agreement_group']))


## 6. Kiểm tra agreement liên tục và xuất kết quả

In [ ]:
from scipy.stats import spearmanr
continuous=[]
for axis,col in [('User-side','user_agreement'),('Item-side','item_agreement_mean_train')]:
    for (model,seed),g in comparison.groupby(['model','seed']):
        x=g[col].to_numpy()
        valid=np.isfinite(x)
        for metric in ['DeltaRecall@20','DeltaNDCG@20']:
            y=g[metric].to_numpy()
            good=valid & np.isfinite(y)
            stat=spearmanr(x[good],y[good]) if good.sum()>3 else None
            continuous.append({'axis':axis,'model':model,'seed':seed,'metric':metric,
                               'n_users':int(good.sum()),
                               'spearman_rho':float(stat.statistic) if stat is not None else np.nan,
                               'p_value_exploratory':float(stat.pvalue) if stat is not None else np.nan})
correlations=pd.DataFrame(continuous)
OUT.mkdir(parents=True,exist_ok=True)
groups.to_csv(OUT/'E3D_agreement_groups.csv',index=False)
seed_summary.to_csv(OUT/'E3D_seed_group_metrics.csv',index=False)
group_summary.to_csv(OUT/'E3D_group_summary.csv',index=False)
correlations.to_csv(OUT/'E3D_continuous_correlations.csv',index=False)
meta={'source_repo':'https://github.com/trungkien1502/project_sna_group14',
      'dataset_root':str(DATA),'analysis':'E3D agreement (sampled anchor Spearman)',
      'anchor_seed':ANCHOR_SEED,'n_anchors':N_ANCHORS,
      'user_anchor_ids':user_anchors.tolist(),'item_anchor_ids':item_anchors.tolist(),
      'user_tertiles':uq.tolist(),'item_tertiles':iq.tolist(),
      'seeds':list(SEEDS),'k':20,'test_users':int(N_TEST_USERS),
      'definition':'same-entity semantic cosine vs train-only interaction cosine, sampled anchors; item-side averaged over train-history items',
      'caveat':'semantic profile generation provenance not independently established train-only; association not causation'}
(OUT/'E3D_method.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
display(correlations)
print('Saved:',OUT)
import shutil
shutil.make_archive('/content/E3D_agreement_results','zip',OUT)
from google.colab import files
files.download('/content/E3D_agreement_results.zip')


### Diễn giải cho báo cáo RQ2

- **User-side:** Nếu `DeltaRecall_mean` ở High lớn hơn Low, lợi ích alignment quan sát được tăng theo user agreement; nếu ngược lại, lợi ích tập trung ở nhóm agreement thấp.
- **Item-side:** Đọc tương tự theo `item_agreement_mean_train`, là agreement trung bình của các sách user đã tương tác ở train.
- **Kiểm tra:** So sánh cả Recall và NDCG, cả 3 seed; không suy luận significance từ một giá trị p thăm dò do các user có thể phụ thuộc nhau.
- **Giới hạn:** Đây là *anchor-based neighborhood agreement proxy*; phụ thuộc lựa chọn anchors, nên có thể chạy lại với `ANCHOR_SEED` khác để kiểm tra độ nhạy. Không chứng minh nhân quả hay loại trừ hoàn toàn data leakage.
